# B1 - Person B, CPU preparation (denoise-study)

**Kaggle session settings:** Accelerator **None** (no GPU quota needed) | Internet **ON** (for `git clone`) |
Inputs: dataset `bsd-data` (pinned version); `denoise-shared` from Person A if it exists (pinned version).

**What this notebook does (in order):** bootstrap repo -> save environment -> `pytest` (incl. torch tests, on CPU)
-> U-Net parameter count -> verify manifest/data hashes and noisy PNGs -> verify committed masks ->
**tune bilateral and TV on the 8 calibration images only** -> review grids (optimum must be interior) ->
(optional, off by default) classical inference.

**Rules this notebook obeys:** no test image is used for any decision here; the notebook is thin (all logic is in
`scripts.*` / `common.*`); nothing FROZEN is edited. Read `PROJECT_CONTEXT.md` Sections 11, 24 and 25 first.

In [1]:
# ---- CONFIG: edit this cell only ----
NOTEBOOK_NAME = "B1_cpu_prep"
REPO_URL = "https://github.com/abhinavpareek655/denoise-study.git"  # repo URL; private repo -> use Kaggle Secrets, never paste a token here.
BRANCH = "main"
PINNED_COMMIT = "db58a0c"            # full commit hash; set it for every run whose results you keep

USE_SHARED_FOR_PIPELINE = False   # True = point DENOISE_SHARED at A's denoise-shared dataset (check common/paths.py first)
RUN_TESTS = True                  # python -m pytest (torch tests run on CPU here)
RUN_MAKE_NOISY_IF_MISSING = True  # deterministic; verified against A's copy below if attached
REGENERATE_MASKS_CHECK = True     # re-run make_masks and confirm the committed thresholds do not change
TUNE_METHODS = ["bilateral", "tv"]  # B owns these two (README stage table)
FORCE_RETUNE = False              # only after widening a grid whose optimum sat on the edge (calibration data only)
RUN_INFER_CLASSICAL = False       # flip to True ONLY after you reviewed and accepted the frozen JSONs below

In [2]:
# ---- Bootstrap (identical in B1 and B2): helpers, repo clone/pin, dataset lookup, env vars ----
import os, sys, json, time, shutil, hashlib, subprocess, datetime, importlib
from pathlib import Path

WORK = Path("/kaggle/working")          # the ONLY directory Kaggle keeps as notebook output
REPO = WORK / "denoise-study"
LOG_DIR = WORK / "notebook_logs"
LOG_DIR.mkdir(exist_ok=True)
assert Path("/kaggle").exists(), "This notebook is written for Kaggle (paths come from /kaggle)."


def sh(cmd, cwd=None, check=True, log=None):
    """Run a shell command, stream output live, optionally tee it to a log file. Returns exit code."""
    print(f"$ {cmd}", flush=True)
    t0 = time.time()
    run_dir = cwd or (REPO if REPO.exists() else None)
    p = subprocess.Popen(cmd, shell=True, cwd=run_dir, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lf = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        if lf:
            lf.write(line)
            lf.flush()
    rc = p.wait()
    if lf:
        lf.close()
    print(f"[exit {rc} after {time.time() - t0:.0f}s]", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc


def find_dir_containing(name, root="/kaggle/input", max_depth=3):
    """Breadth-first search under `root` for a directory that contains `name`."""
    queue = [(Path(root), 0)]
    while queue:
        d, depth = queue.pop(0)
        if not d.is_dir():
            continue
        if (d / name).exists():
            return d
        if depth < max_depth:
            queue.extend((c, depth + 1) for c in sorted(d.iterdir()) if c.is_dir())
    return None


# --- code: clone from GitHub (Internet ON) or copy from an attached code dataset (Internet OFF) ---
if not REPO.exists():
    if REPO_URL:
        assert "<" not in REPO_URL, "Edit REPO_URL in the config cell first."
        sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO}", cwd=WORK)
    else:
        src = find_dir_containing("PROJECT_CONTEXT.md")
        assert src is not None, "No REPO_URL and no attached code dataset containing PROJECT_CONTEXT.md"
        shutil.copytree(src, REPO)
if (REPO / ".git").exists() and REPO_URL:
    sh("git fetch --all --prune")
    sh(f"git checkout {PINNED_COMMIT}" if PINNED_COMMIT else f"git checkout {BRANCH} && git pull --ff-only")
COMMIT = subprocess.run("git rev-parse HEAD", shell=True, cwd=REPO, capture_output=True,
                        text=True).stdout.strip() or "no-git"
print("code commit:", COMMIT)

# --- data: the attached bsd-data dataset must contain BSD68/ and BSD400/ ---
DATA = find_dir_containing("BSD68")
assert DATA is not None and (DATA / "BSD400").exists(), "Attach the bsd-data dataset (BSD68/ and BSD400/)."
os.environ["DENOISE_DATA"] = str(DATA)
SHARED_DIR = find_dir_containing("manifest.json")      # A's denoise-shared dataset, if attached
if USE_SHARED_FOR_PIPELINE:
    assert SHARED_DIR is not None, "USE_SHARED_FOR_PIPELINE=True but no denoise-shared dataset is attached"
    os.environ["DENOISE_SHARED"] = str(SHARED_DIR)
print("DENOISE_DATA  =", os.environ["DENOISE_DATA"])
print("DENOISE_SHARED=", os.environ.get("DENOISE_SHARED", "(unset: repo artifacts/ is used)"))
print("shared dataset found at:", SHARED_DIR)

os.chdir(REPO)
sys.path.insert(0, str(REPO))
paths = importlib.import_module("common.paths")
print("\ncommon.paths resolves (CHECK these look right before running anything):")
for n in sorted(dir(paths)):
    v = getattr(paths, n)
    if not n.startswith("_") and isinstance(v, (str, Path, dict)):
        print(f"  {n} = {v}")
print("  functions:", [n for n in dir(paths) if not n.startswith("_") and callable(getattr(paths, n))])


$ git clone --branch main https://github.com/abhinavpareek655/denoise-study.git /kaggle/working/denoise-study
Cloning into '/kaggle/working/denoise-study'...
[exit 0 after 1s]
$ git fetch --all --prune
[exit 0 after 0s]
$ git checkout db58a0c
Note: switching to 'db58a0c'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at db58a0c Add Person B notebooks and repo cleanup
[exit 0 after 0s]
code commit: db58a0cbafcf892fcf054729d4bbe0190b0fbd76
DENOISE_DATA  = /kaggle/input/datasets/abhinavpareek655/bsd-data
DENOISE_SH

In [3]:
# ---- Environment capture (Section 24.3): saved to artifacts/logs/env_<timestamp>.json ----
import platform
import numpy, scipy, skimage, pandas


def _cap(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60)
    return r.stdout.strip()


env = {
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "notebook": NOTEBOOK_NAME,
    "code_commit": COMMIT,
    "python": platform.python_version(),
    "numpy": numpy.__version__, "scipy": scipy.__version__,
    "scikit_image": skimage.__version__, "pandas": pandas.__version__,
    "cpu_model": _cap("lscpu | grep 'Model name' | head -1 | cut -d: -f2"),
    "cpu_count": os.cpu_count(),
    "gpu": _cap("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") or "none",
    "data_dir": str(DATA), "shared_dir": str(SHARED_DIR),
}
try:
    import torch
    env.update(torch=torch.__version__, cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except Exception as e:  # torch missing is reported, not fatal, in the CPU notebook
    env["torch"] = f"unavailable: {e!r}"
(REPO / "artifacts" / "logs").mkdir(parents=True, exist_ok=True)
env_path = REPO / "artifacts" / "logs" / f"env_{datetime.datetime.now():%Y%m%d_%H%M%S}.json"
env_path.write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

# The contract wants common/versions.py output here; its API is not in the notes, so list what exists:
try:
    vers = importlib.import_module("common.versions")
    print("common.versions public names:", [n for n in dir(vers) if not n.startswith("_")])
except Exception as e:
    print("common.versions failed to import:", repr(e))


{
  "timestamp_utc": "2026-11-14T05:45:24.126651+00:00",
  "notebook": "B1_cpu_prep",
  "code_commit": "db58a0cbafcf892fcf054729d4bbe0190b0fbd76",
  "python": "3.13.18",
  "numpy": "2.1.3",
  "scipy": "1.16.3",
  "scikit_image": "0.25.2",
  "pandas": "2.3.3",
  "cpu_model": "Intel(R) Xeon(R) CPU @ 2.20GHz",
  "cpu_count": 4,
  "gpu": "none",
  "data_dir": "/kaggle/input/datasets/abhinavpareek655/bsd-data",
  "shared_dir": "None",
  "torch": "2.11.0+cpu",
  "cuda": null,
  "cuda_available": false
}
common.versions public names: ['annotations', 'env_info', 'git_commit', 'hardware_string', 'platform', 'subprocess']


## 1. Tests - the first action on Kaggle
The torch code (models, losses, trainer, resume test) has **never been executed** (README, "Status of this scaffold").
Expect failures here; fix them in the repo on your laptop, push, and re-run with the new commit. Do not patch files in `/kaggle/working`.

In [4]:
if RUN_TESTS:
    sh("python -m pytest -q 2>&1 | tail -n 60", log=LOG_DIR / "pytest_B1.log", check=False)
else:
    print("skipped (RUN_TESTS=False)")

$ python -m pytest -q 2>&1 | tail -n 60
.............................................                            [100%]
[exit 0 after 38s]


In [5]:
# U-Net / DnCNN parameter counts. Expected (hand-computed, Section 25.2 item 7): U-Net 482,449 ; DnCNN 17x64 ~556,096.
# Put the REAL numbers into docs/model_provenance.md. The full model is the largest class printed per module.
import inspect
import torch.nn as nn


def count_params(m):
    return sum(p.numel() for p in m.parameters())


for modname in ("models.unet", "models.dncnn"):
    try:
        mod = importlib.import_module(modname)
        rows = []
        for name, cls in inspect.getmembers(mod, inspect.isclass):
            if issubclass(cls, nn.Module) and cls.__module__ == modname:
                try:
                    rows.append((count_params(cls()), name))
                except Exception as e:
                    rows.append((-1, f"{name} (needs args: {e})"))
        for n, name in sorted(rows, reverse=True):
            print(f"{modname}.{name}: {n:,} params")
    except Exception as e:
        print(modname, "->", repr(e))

models.unet.ResUNet: 482,449 params
models.dncnn.DnCNN: 556,096 params


## 2. Verify inputs: manifest, raw data, noisy PNGs, masks
Checks counts 8/60/380/20, hashes every raw image against the manifest, and confirms the noisy PNGs are
byte-identical to A's `denoise-shared` copy (if attached). Both people must evaluate against the same manifest hash.

In [6]:
from collections import Counter
from common import protocol as PR

man_path = REPO / "artifacts" / "manifest.json"
man = json.loads(man_path.read_text())
print("manifest sha256:", hashlib.sha256(man_path.read_bytes()).hexdigest())
cnt = Counter(r["split"] for r in man["images"])
print(dict(cnt))
assert cnt == {"calibration": PR.N_CALIBRATION, "test": PR.N_TEST, "train": PR.N_TRAIN, "validation": PR.N_VAL}, cnt

bad = []
for r in man["images"]:
    rel = Path(r["relative_path"])
    parts = rel.parts[1:] if rel.parts and rel.parts[0] == "data" else rel.parts
    f = DATA.joinpath(*parts)
    if not f.exists() or hashlib.sha256(f.read_bytes()).hexdigest() != r["sha256"]:
        bad.append(r["image_id"])
print("images failing the manifest hash check:", bad if bad else "none")
assert not bad, "raw data differ from the frozen manifest: wrong bsd-data version?"

manifest sha256: 6a820cc7263625a0325b0a1bc5a587ffcb739a9636549942f9e1366f452f06e1
{'test': 60, 'calibration': 8, 'train': 380, 'validation': 20}
images failing the manifest hash check: none


In [7]:
NOISY = REPO / "artifacts" / "noisy"
expected = PR.N_CALIBRATION + PR.N_TEST          # validation noise is regenerated in memory (Section 25.2 item 5)


def n_noisy():
    return {S: len(list((NOISY / f"sigma{S}").glob("*.png"))) for S in PR.SIGMAS_255}


have = n_noisy()
print("noisy PNGs per sigma:", have, "| expected", expected)
if any(v < expected for v in have.values()) and RUN_MAKE_NOISY_IF_MISSING:
    sh("python -m scripts.make_noisy", log=LOG_DIR / "make_noisy.log")
    have = n_noisy()
    print("after make_noisy:", have)
assert all(v >= expected for v in have.values()), "noisy sets incomplete"

shared_noisy = find_dir_containing("sigma15")        # A's copy, if the dataset is attached
if shared_noisy is None:
    print("No denoise-shared noisy set attached: cannot cross-check against A (do this before final runs).")
else:
    mism = []
    for S in PR.SIGMAS_255:
        for f in sorted((shared_noisy / f"sigma{S}").glob("*.png")):
            g = NOISY / f"sigma{S}" / f.name
            if not g.exists() or hashlib.sha256(g.read_bytes()).hexdigest() != hashlib.sha256(f.read_bytes()).hexdigest():
                mism.append((S, f.name))
    print("noisy PNG mismatches vs A:", mism[:10] if mism else "none", f"({len(mism)} total)")
    assert not mism, "your noisy PNGs differ from A's: stop and compare GLOBAL_SEED / manifest / code commit"

noisy PNGs per sigma: {15: 0, 25: 0, 50: 0} | expected 68
$ python -m scripts.make_noisy
noisy 15 {'n': 68, 'preclip_mean_avg': -0.004859251493827136, 'preclip_std_avg': 14.9940456655318}
noisy 25 {'n': 68, 'preclip_mean_avg': 0.000286534781570181, 'preclip_std_avg': 25.003240480670684}
noisy 50 {'n': 68, 'preclip_mean_avg': 0.018272312603409748, 'preclip_std_avg': 50.00287956098121}
[exit 0 after 5s]
after make_noisy: {15: 68, 25: 68, 50: 68}
No denoise-shared noisy set attached: cannot cross-check against A (do this before final runs).


In [8]:
th = REPO / "artifacts" / "masks" / "thresholds.json"
print(th.read_text()[:1500])
print("pixel_counts.csv present:", (REPO / "artifacts" / "masks" / "pixel_counts.csv").exists())
if REGENERATE_MASKS_CHECK:
    rc = sh("python -m scripts.make_masks --force", log=LOG_DIR / "make_masks.log", check=False)
    if rc != 0:
        print("make_masks rejected --force (see 'python -m scripts.make_masks --help'); skipping the reproducibility check")
    else:
        sh("git status --short artifacts/masks; git diff --stat -- artifacts/masks", check=False)
        print("Empty git output above = masks/thresholds reproduce exactly.")

{
 "default": {
  "grad_smooth": 0.008727212527841643,
  "std_smooth": 0.026975421643376706,
  "grad_edge": 0.059738787617757465,
  "std_texture": 0.052696889909743846,
  "pcts": {
   "smooth": 40,
   "edge": 90,
   "texture": 60
  }
 },
 "low": {
  "grad_smooth": 0.005951985682406885,
  "std_smooth": 0.01918579988998057,
  "grad_edge": 0.03737099135421041,
  "std_texture": 0.03728856914169414,
  "pcts": {
   "smooth": 30,
   "edge": 80,
   "texture": 50
  }
 },
 "high": {
  "grad_smooth": 0.012414724820583218,
  "std_smooth": 0.03728856914169414,
  "grad_edge": 0.14524392521963045,
  "std_texture": 0.07290938708463163,
  "pcts": {
   "smooth": 50,
   "edge": 99,
   "texture": 70
  }
 },
 "fitted_on": [
  "test027",
  "test043",
  "test049",
  "test054",
  "test056",
  "test059",
  "test060",
  "test065"
 ]
}

pixel_counts.csv present: True
$ python -m scripts.make_masks --force
         n_smooth   n_edge  n_texture
variant                              
default   45858.0  19079.0    49

## 3. Tune bilateral and TV (calibration images only, per sigma)
`scripts.tune` writes `artifacts/tuning/{method}_sigma{S}_grid.csv` and `{method}_frozen.json`.
**Frozen settings are final**: no retuning after any test result is seen. The one allowed reason to re-tune is a
grid optimum on the grid edge (Section 25.2 item 8) - widen using calibration data only, then `FORCE_RETUNE=True`.

In [9]:
sh("python -m scripts.tune --help | head -n 25", check=False)
for m in TUNE_METHODS:
    t0 = time.time()
    sh(f"python -m scripts.tune --method {m}" + (" --force" if FORCE_RETUNE else ""),
       log=LOG_DIR / f"tune_{m}.log")
    print(f"{m}: {(time.time() - t0) / 60:.1f} min")

$ python -m scripts.tune --help | head -n 25
usage: tune.py [-h] --method METHOD [--grid GRID]

options:
  -h, --help       show this help message and exit
  --method METHOD
  --grid GRID
[exit 0 after 1s]
$ python -m scripts.tune --method bilateral
bilateral sigma=15: best sigma_r_255=35;sigma_s_px=1.5 -> 30.00 dB
bilateral sigma=25: best sigma_r_255=75;sigma_s_px=1.5 -> 27.37 dB
bilateral sigma=50: best sigma_r_255=170;sigma_s_px=2 -> 24.25 dB
[exit 0 after 250s]
bilateral: 4.2 min
$ python -m scripts.tune --method tv
tv sigma=15: best eps=0.0002;max_num_iter=200;weight=0.04 -> 30.21 dB
tv sigma=25: best eps=0.0002;max_num_iter=200;weight=0.08 -> 27.78 dB
tv sigma=50: best eps=0.0002;max_num_iter=200;weight=0.15 -> 24.69 dB
[exit 0 after 15s]
tv: 0.2 min


In [10]:
import pandas as pd

META_HINTS = ("psnr", "ssim", "mae", "runtime", "time", "sigma_255", "n_", "warn", "image", "split", "seed")


def review_tuning(method):
    """Print frozen settings, best grid row per sigma, and flag optima that sit on a grid edge."""
    tdir = REPO / "artifacts" / "tuning"
    fz = tdir / f"{method}_frozen.json"
    print(f"\n==================== {method}: frozen settings ====================")
    print(fz.read_text() if fz.exists() else f"MISSING {fz}")
    for S in PR.SIGMAS_255:
        f = tdir / f"{method}_sigma{S}_grid.csv"
        if not f.exists():
            print("MISSING", f)
            continue
        df = pd.read_csv(f)
        metric = next((c for c in df.columns if "psnr" in c.lower()), None)
        if metric is None:
            print(f"sigma {S}: no PSNR column; columns = {list(df.columns)}")
            continue
        best = df.loc[df[metric].idxmax()]
        print(f"\n-- sigma {S}: {len(df)} grid rows, best {metric} = {best[metric]:.3f} dB")
        params = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])
                  and not any(h in c.lower() for h in META_HINTS) and df[c].nunique() >= 3]
        for c in params:
            edge = best[c] in (df[c].min(), df[c].max())
            print(f"   {'WARNING edge ' if edge else 'ok interior'}: {c} = {best[c]}"
                  + ("  -> widen the grid (calibration only) and re-tune" if edge else ""))
        print(df.sort_values(metric, ascending=False).head(5).to_string(index=False))


for m in TUNE_METHODS:
    review_tuning(m)
print("\nYardstick (Section 18): best classical on calibration crops was roughly 29.3 / 26.3 / 22.6 dB at sigma 15/25/50;"
      " noisy input is about 24.6 / 20.2 / 14.2 dB. A far-off number means a scale or grid bug.")


==================== bilateral: frozen settings ====================
{
 "15": {
  "config": {
   "sigma_s_px": 1.5,
   "sigma_r_255": 35.0
  },
  "parameter_config": "sigma_r_255=35;sigma_s_px=1.5",
  "calibration_mean_psnr": 30.001896117499
 },
 "25": {
  "config": {
   "sigma_s_px": 1.5,
   "sigma_r_255": 75.0
  },
  "parameter_config": "sigma_r_255=75;sigma_s_px=1.5",
  "calibration_mean_psnr": 27.368796917416795
 },
 "50": {
  "config": {
   "sigma_s_px": 2.0,
   "sigma_r_255": 170.0
  },
  "parameter_config": "sigma_r_255=170;sigma_s_px=2",
  "calibration_mean_psnr": 24.2494065488668
 },
 "_meta": {
  "method": "bilateral",
  "tuned_on": [
   "test027",
   "test043",
   "test049",
   "test054",
   "test056",
   "test059",
   "test060",
   "test065"
  ],
  "objective": "mean quantized PSNR on calibration images",
  "manifest_images_sha256": "f52ced08cdccacac61c3c9813c63558732a5283e75886cb3d40501e4b1940bc7"
 }
}


-- sigma 15: 20 grid rows, best mean_psnr = 30.002 dB
             p

## 4. Classical inference on the 60 test images (optional, **off by default**)
Only after you reviewed the frozen JSONs above. This is the first time test images are touched by your methods;
nothing here may feed back into tuning. Evaluation/merging with A's outputs is a later notebook (Days 8-10).

In [11]:
if RUN_INFER_CLASSICAL:
    for m in TUNE_METHODS:
        sh(f"python -m scripts.infer --method {m}", log=LOG_DIR / f"infer_{m}.log")
else:
    print("skipped (RUN_INFER_CLASSICAL=False)")

skipped (RUN_INFER_CLASSICAL=False)


In [12]:
def du_mb(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1e6


for sub in ("tuning", "masks", "logs", "noisy", "outputs", "figures"):
    p = REPO / "artifacts" / sub
    if p.exists():
        print(f"artifacts/{sub:9s} {du_mb(p):8.1f} MB")
print(f"/kaggle/working total: {du_mb(WORK):.0f} MB   (limit is about 20 GB)")

artifacts/tuning         0.0 MB
artifacts/masks          0.0 MB
artifacts/logs           0.0 MB
artifacts/noisy         28.6 MB
artifacts/outputs        0.0 MB
artifacts/figures        0.0 MB
/kaggle/working total: 29 MB   (limit is about 20 GB)


## After the run (do these, they are not automatic)
1. **Save Version -> Save & Run All (Commit)** so `/kaggle/working` is stored as notebook output.
2. From the output, copy `artifacts/tuning/*` (and `artifacts/logs/env_*.json`) back into the repo on your laptop; commit
   `bilateral_frozen.json` / `tv_frozen.json` and the grid CSVs; push. Both are small result files that belong in git.
3. Create/Update the dataset **`denoise-B-results`** from this notebook's output; **write down the version number**.
4. Add a dated line to `PROJECT_CONTEXT.md` Section 22 (frozen settings, commit hash, dataset version) and tell A.